# Ensemble Methods Compared: Random Forest vs Gradient Boosting vs XGBoost

Three tree **ensembles** trained on the *identical* train/test split, judged on three axes:

1. **Predictive quality** &mdash; test accuracy and ROC AUC.
2. **Cost** &mdash; wall-clock training time via `time.perf_counter`.
3. **Interpretability** &mdash; which features each model leans on.

The three models split into two families:

| Model | Family | One-line idea |
|-------|--------|---------------|
| `RandomForestClassifier` | **Bagging** | many *independent* deep trees on bootstrap samples, then vote/average |
| `GradientBoostingClassifier` | **Boosting** | many *shallow* trees added in sequence, each fixing the last one's errors |
| `XGBoost` (or `HistGradientBoosting` stand-in) | **Boosting** | the same boosting idea, engineered to be fast and heavily regularizable |

The comparison below makes the classic trade-off visible: boosting usually edges out bagging on accuracy, but it trains *sequentially* (slower) and has more knobs to tune.

## A note on installing XGBoost

XGBoost is a separate library, not part of scikit-learn. In a normal environment you would install it with:

```bash
pip install xgboost
```

**This environment is offline and XGBoost is not installed**, so the very next code cell tries to import it and, if that fails, transparently falls back to scikit-learn's `HistGradientBoostingClassifier`. That fallback is an excellent XGBoost stand-in: it is also a histogram-based gradient-boosted-trees model (the same core algorithm XGBoost popularized), so the comparison stays fair. Every place we use it, we clearly label it as the *XGBoost stand-in*.

In [ ]:
import time                                             # perf_counter for precise training-time measurement
import warnings                                         # we silence expected, harmless warnings for a clean run
import numpy as np                                      # arrays / seeding
import pandas as pd                                     # the results table and tidy feature-importance frames
import matplotlib.pyplot as plt                         # plotting
import seaborn as sns                                   # nicer bar charts for feature importances

from sklearn.datasets import load_breast_cancer         # a clean, named, binary classification dataset
from sklearn.model_selection import train_test_split    # single split shared by all three models
from sklearn.ensemble import (
    RandomForestClassifier,                             # the BAGGING model
    GradientBoostingClassifier,                         # the classic BOOSTING model
    HistGradientBoostingClassifier,                     # fast histogram booster -> our XGBoost stand-in
)
from sklearn.metrics import accuracy_score, roc_auc_score  # the two quality metrics we report
from sklearn.inspection import permutation_importance   # importance for models lacking .feature_importances_

# One global seed reused everywhere so the whole notebook is byte-for-byte reproducible.
SEED = 42
np.random.seed(SEED)                                    # legacy global RNG (belt-and-suspenders)
sns.set_theme(style="whitegrid")                        # consistent, readable plot styling
warnings.filterwarnings("ignore")                       # keep the output free of expected convergence chatter

print("imports ready")

### Import XGBoost if available, otherwise fall back

The pattern below is the standard defensive import. `XGB_AVAILABLE` records which path we took so later cells can label the model honestly, and `make_xgb()` hides the difference behind one factory function &mdash; the rest of the notebook never has to care which class it actually got.

In [ ]:
try:
    # The happy path: real XGBoost is installed.
    from xgboost import XGBClassifier                   # noqa: F401
    XGB_AVAILABLE = True
    XGB_LABEL = "XGBoost"                                # honest label for tables/plots

    def make_xgb():
        """Build a modestly-sized XGBoost classifier with a fixed seed."""
        return XGBClassifier(
            n_estimators=200,          # number of boosting rounds (trees added in sequence)
            max_depth=3,               # shallow trees = weak learners, the boosting norm
            learning_rate=0.1,         # shrinkage: how much each new tree is trusted
            subsample=0.9,             # row subsampling per tree -> a little regularization
            random_state=SEED,
            eval_metric="logloss",     # silence the default-metric warning on modern XGBoost
            n_jobs=1,                  # single-threaded keeps timing comparable + deterministic
        )

except ImportError:
    # The fallback path (this is what actually runs here): no xgboost -> use HistGradientBoosting.
    XGB_AVAILABLE = False
    XGB_LABEL = "HistGB (XGBoost stand-in)"             # label makes the substitution obvious everywhere

    def make_xgb():
        """Build sklearn's histogram gradient booster as a drop-in XGBoost substitute."""
        return HistGradientBoostingClassifier(
            max_iter=200,              # analogue of n_estimators (boosting rounds)
            max_depth=3,               # keep trees shallow, matching the XGBoost config above
            learning_rate=0.1,         # same shrinkage idea
            random_state=SEED,
        )

# Report which branch we landed on so the reader is never confused about what 'XGB' means below.
print(f"xgboost installed? {XGB_AVAILABLE}")
print(f"the third model will be labelled: '{XGB_LABEL}'")

## 1. Data: Breast Cancer Wisconsin

A binary classification task: predict whether a tumor is **malignant** or **benign** from 30 numeric features (cell-nucleus measurements like `mean radius`, `worst texture`, ...). Binary labels make **ROC AUC** meaningful, and the features carry human-readable names, which is what makes the feature-importance comparison interesting.

Tree ensembles are **scale-invariant** (they split on thresholds, not distances), so &mdash; unlike distance-based models &mdash; we deliberately do *not* standardize the features. We only do a stratified train/test split so every model sees the exact same data.

In [ ]:
data = load_breast_cancer()                # Bunch with .data, .target, .feature_names, .target_names
X = pd.DataFrame(data.data, columns=data.feature_names)  # keep feature NAMES by wrapping in a DataFrame
y = data.target                            # 0 = malignant, 1 = benign (sklearn's encoding)
feature_names = [str(f) for f in data.feature_names]  # plain-str names, reused for every importance plot

# ONE split, shared by all three models -> the comparison is apples-to-apples.
# stratify=y preserves the malignant/benign ratio in both halves; test_size=0.25 holds out 25%.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# np.bincount gives counts per class label; zip with the readable class names for a tidy print.
class_counts = {name: int(n) for name, n in zip(data.target_names, np.bincount(y))}
print(f"features           : {X.shape[1]}")
print(f"train / test sizes : {X_train.shape[0]} / {X_test.shape[0]}")
print(f"classes            : {class_counts}")
X_train.head(3)                            # peek at a few named rows

## 2. Bagging vs Boosting &mdash; the core idea

Both build a *forest* of decision trees, but they combine them in opposite ways.

### Bagging (Random Forest)
Train $T$ **deep, independent** trees, each on a different bootstrap resample of the data and each considering only a random subset of features at every split. Because the trees are decorrelated, **averaging** their votes cancels out their individual variance:

$$\hat{y} = \frac{1}{T}\sum_{t=1}^{T} f_t(x)$$

Each tree overfits on its own (high variance, low bias); the average is stable. Training is **embarrassingly parallel** &mdash; the trees never talk to each other.

### Boosting (Gradient Boosting / XGBoost)
Train $T$ **shallow, weak** trees **in sequence**. Each new tree $h_t$ is fit to the *errors* (negative gradient of the loss) left by the running ensemble so far, and is added with a small learning rate $\eta$ (shrinkage):

$$F_t(x) = F_{t-1}(x) + \eta \, h_t(x)$$

Boosting attacks **bias**: it keeps chipping away at the residual mistakes. That sequential dependence is exactly why it usually reaches higher accuracy &mdash; and exactly why it **cannot be parallelized across trees** and needs more careful tuning (`learning_rate`, `n_estimators`, `max_depth` all interact). Too many rounds and it starts fitting noise; $\eta$ and depth are the main dials that trade fit against overfitting.

**Rule of thumb:** Random Forest is the robust, low-effort baseline; a well-tuned booster is the higher-ceiling but higher-maintenance option.

## 3. Train all three, timing each

We keep every model **modest** (few hundred trees, shallow where it's a booster) so the whole notebook finishes in seconds. `time.perf_counter()` is the right clock here: it is a high-resolution monotonic timer meant for measuring short durations, unlike `time.time()` which can jump if the system clock is adjusted.

In [ ]:
# Instantiate the three contenders. Sizes are deliberately small for a fast, offline run.
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,      # 200 independent trees
        max_depth=None,        # let each tree grow deep -> high variance, tamed by averaging (bagging)
        random_state=SEED,
        n_jobs=1,              # single thread so training time is comparable to the sequential boosters
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,      # 200 boosting rounds (trees added one after another)
        max_depth=3,           # shallow weak learners, the boosting convention
        learning_rate=0.1,     # shrinkage on each round
        random_state=SEED,
    ),
    XGB_LABEL: make_xgb(),      # real XGBoost OR the HistGB stand-in -> label was chosen in the import cell
}

# Train each model, recording fitted estimator + how long .fit() took.
fitted = {}          # name -> trained model
train_times = {}     # name -> seconds spent in .fit()

for name, model in models.items():
    start = time.perf_counter()          # high-resolution clock: start
    model.fit(X_train, y_train)          # THE training step we are timing
    elapsed = time.perf_counter() - start  # seconds elapsed for this fit
    fitted[name] = model
    train_times[name] = elapsed
    print(f"trained {name:<28s} in {elapsed:6.3f} s")

## 4. Results table: accuracy, ROC AUC, and training time

- **Accuracy** = fraction of correct malignant/benign calls at the 0.5 threshold.
- **ROC AUC** = probability the model ranks a random positive above a random negative; it uses the predicted *probabilities* (`predict_proba`), so it is threshold-independent and a finer discriminator than accuracy.
- **Train time** = the sequential cost we measured above.

All three metrics are computed on the **same held-out test set**.

In [ ]:
rows = []
for name, model in fitted.items():
    y_pred = model.predict(X_test)                      # hard 0/1 predictions -> for accuracy
    y_proba = model.predict_proba(X_test)[:, 1]         # P(class 1) -> for ROC AUC (needs probabilities)
    rows.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "train_time_s": train_times[name],
    })

# Build a tidy DataFrame, sorted by ROC AUC (best first) for easy reading.
results = pd.DataFrame(rows).set_index("model").sort_values("test_roc_auc", ascending=False)
results_display = results.copy()
results_display["test_accuracy"] = results_display["test_accuracy"].map("{:.4f}".format)
results_display["test_roc_auc"] = results_display["test_roc_auc"].map("{:.4f}".format)
results_display["train_time_s"] = results_display["train_time_s"].map("{:.3f}".format)
print(results_display.to_string())
results_display

In [ ]:
# Visualize the trade-off: quality on the left, cost on the right.
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Left: accuracy and ROC AUC as grouped bars ---
idx = np.arange(len(results))            # one group position per model
bar_w = 0.38                             # width of each bar within a group
ax[0].bar(idx - bar_w / 2, results["test_accuracy"], bar_w, label="accuracy", color="#4c72b0")
ax[0].bar(idx + bar_w / 2, results["test_roc_auc"], bar_w, label="roc_auc", color="#dd8452")
ax[0].set_xticks(idx)
ax[0].set_xticklabels(results.index, rotation=15, ha="right")
ax[0].set_ylim(0.90, 1.005)              # zoom in: all models are strong, differences are small
ax[0].set_title("Predictive quality (higher = better)")
ax[0].legend(loc="lower right")

# --- Right: training time (note this is where boosting typically pays a price) ---
ax[1].bar(results.index, results["train_time_s"], color="#55a868")
ax[1].set_xticklabels(results.index, rotation=15, ha="right")
ax[1].set_ylabel("seconds")
ax[1].set_title("Training time (lower = cheaper)")
for i, v in enumerate(results["train_time_s"]):          # annotate each bar with its value
    ax[1].text(i, v, f"{v:.3f}s", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 5. Feature importance, three ways

**How to read it.** A feature-importance score answers *"how much did the model rely on this feature?"* &mdash; higher means more influential. It is **not** a causal claim and it says nothing about the *direction* of an effect; it only ranks how useful each column was for splitting.

Two flavors appear here:

- **Impurity-based** (`.feature_importances_`): for Random Forest and Gradient Boosting, this is how much each feature reduced impurity (Gini / loss) across all splits, summed and normalized to 1. Fast (it's a by-product of training) but biased toward high-cardinality / continuous features.
- **Permutation importance**: `HistGradientBoostingClassifier` does **not** expose `.feature_importances_`, so for the XGBoost stand-in we instead **shuffle one feature at a time** on the test set and measure how much the score drops. A big drop = the model depended on that feature. This is model-agnostic but costs extra compute (many re-scorings). *(Real XGBoost does have `.feature_importances_`; the code below picks the right method automatically.)*

In [ ]:
def get_importances(name, model):
    """Return (importance_vector, method_label) for a fitted model.

    Uses built-in .feature_importances_ when available; otherwise falls back to
    permutation importance (the HistGradientBoosting stand-in path).
    """
    if hasattr(model, "feature_importances_"):
        # RandomForest, GradientBoosting, and real XGBoost all land here.
        return np.asarray(model.feature_importances_), "impurity"
    else:
        # HistGradientBoosting lacks .feature_importances_ -> permutation importance instead.
        # n_repeats kept small for speed; scoring on the TEST set measures generalization reliance.
        result = permutation_importance(
            model, X_test, y_test,
            n_repeats=10, random_state=SEED, scoring="roc_auc", n_jobs=1,
        )
        return result.importances_mean, "permutation"

# Collect an importance vector + its method label for each model.
importances = {}   # name -> (vector, method)
for name, model in fitted.items():
    vec, method = get_importances(name, model)
    importances[name] = (vec, method)
    print(f"{name:<28s} -> {method} importance ({len(vec)} features)")

In [ ]:
# Plot each model's TOP-10 features side by side so we can compare what each one relies on.
TOP_N = 10
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=False)

for ax, (name, (vec, method)) in zip(axes, importances.items()):
    # Rank features by importance and keep the strongest TOP_N.
    order = np.argsort(vec)[::-1][:TOP_N]               # indices of the largest scores, descending
    top_feats = [feature_names[i] for i in order]
    top_vals = vec[order]

    # Horizontal bars, most important on top (argsort already gave descending order).
    sns.barplot(x=top_vals, y=top_feats, ax=ax, color="#4c72b0")
    ax.set_title(f"{name}\n({method} importance)", fontsize=11)
    ax.set_xlabel("importance")

plt.suptitle("Top-10 feature importances per model", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

# Because the scales differ (impurity sums to 1; permutation is an AUC drop), print the
# top-5 feature NAMES per model too -- the RANKING is what's comparable across methods, not the raw numbers.
print("Top-5 features by model (ranking is the comparable part, not the raw scale):\n")
for name, (vec, method) in importances.items():
    top5 = [feature_names[i] for i in np.argsort(vec)[::-1][:5]]
    print(f"{name:<28s}: {top5}")

## 6. Takeaways

- **Quality.** On this well-behaved dataset all three ensembles are strong (accuracy and AUC in the mid-0.9s). Boosting typically edges out the Random Forest on ROC AUC because it directly minimizes a differentiable loss, round after round.
- **Cost.** Random Forest trains its trees independently (and could parallelize across cores), whereas the boosters are inherently **sequential** &mdash; each tree waits for the previous one. That is the price of boosting's higher ceiling. `HistGradientBoosting` / XGBoost claw much of that back with histogram binning, which is why they are the go-to boosters on larger data.
- **Interpretability.** The models broadly agree on the *dominant* predictors (the `worst`-suffixed size/shape measurements), a reassuring sign that the signal is real and not an artifact of one algorithm. Remember: impurity importance and permutation importance are on **different scales**, so compare the *rankings*, not the raw magnitudes.
- **Practical advice.** Reach for a Random Forest as a fast, hard-to-mess-up baseline. Move to a tuned gradient booster (XGBoost / LightGBM / HistGB) when you need the last few points of accuracy and are willing to tune `learning_rate` &times; `n_estimators` &times; `max_depth` and watch for overfitting.

*Reminder: XGBoost was unavailable in this offline environment, so the third model was scikit-learn's `HistGradientBoostingClassifier` acting as a faithful stand-in. Install `xgboost` and re-run to swap in the real thing &mdash; the factory function makes it automatic.*